# 02 The call figures in the reports up to the end of 2024

Each of the 39 targets was searched in the UBS quarterly, annual and Pillar 3 reports up to the end of 2024, with terms fixed from the call sentence before the search. A report sentence counts only when it states the quantity as a target, plan or expectation: class A with the same figure, class B with another or no figure, class C not found.

In [1]:
# project setup - run first, do not edit except NAME
NAME = "call_only_targets"

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}")

ok  python 3.13.15  pandas 2.2.3


In [2]:
# folders and helpers shared by the notebooks of this section
import json, re
NB_DIR = ROOT / "notebooks" / "05_call_only_targets"
sys.path.insert(0, str(NB_DIR))
import report_checks as rc

IN = DATA / "call_only_targets"        # delivered result files
REPORTS = DATA / "reports"             # report files, optional (see 00_get_reports)
EXPORTS = NB_DIR / "exports"
EXPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160, "display.max_rows", 60)

manifest = pd.read_csv(IN / "reports_manifest.csv", keep_default_na=False)
status = rc.report_status(REPORTS, manifest)
print(rc.mode_message(status))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.


## Classes

In [3]:
targets = pd.read_csv(IN / "targets_39.csv", keep_default_na=False)
classes = targets.groupby(["class_to_2024", "report_status"]).size().rename("targets").reset_index()
classes

,class_to_2024,report_status,targets
0,A,same figure,23
1,B,another figure,7
2,B,no figure,4
3,B,separate,1
4,C,not mentioned,4


Of the 39 targets, 23 have the same figure in a report, 7 another figure, 4 are mentioned without a figure and 4 are not mentioned. The total of integration-related expenses (T30) is counted separately.

## The eight figures stated only in the calls

In [4]:
call_only = targets[targets.report_status.isin(["not mentioned", "no figure"])]
call_only[["unit", "call", "figure_call", "report_status", "figure_report"]].sort_values(["report_status", "unit"])

,unit,call,figure_call,report_status,figure_report
36,F12,2023-Q4,leverage ratio denominator down by over 100bn over three years,no figure,
0,T16,2023-Q3,NCL operational risk RWA around 14bn by end-2026,no figure,"NCL key priority: reduce RWA and LRD, no figure"
1,T25,2023-Q4,NCL capital release over 6bn by end-2026,no figure,"capital released from the NCL unwinding, no figure"
2,T27,2023-Q4,NCL credit and market risk RWA substantially below 40bn by end-2024,no figure,"NCL run-down offsets Group RWA effects in 2024 and 2025, no figure"
35,F11,2023-Q4,going concern capital ratio around 18% by building out AT1,not mentioned,
38,F22,2024-Q4,LCR below the 4Q24 level of 188% going forward,not mentioned,
4,T33,2023-Q4,AT1 issuance up to 2bn in 2024,not mentioned,
5,T68,2024-Q4,Group HoldCo around 90bn by end-2025,not mentioned,


Eight targets have a figure that no report up to the end of 2024 states as a target: four are not mentioned at all and four are mentioned without a figure. For T25 and F12 the 2024 annual report states a result of the size of the call figure after the fact (see `04_reports_2025_2026.ipynb`).

In [5]:
call_only[["unit", "call_sentence", "quote", "file", "page"]]

,unit,call_sentence,quote,file,page
0,T16,"On the basis of natural roll-off in both contexts, we expect operational risk RWA in NCL to decrease to around 14 billion by the end of 2026.","Reduce RWA and LRD, freeing up capital for the UBS Group",UBS_2023_annual_report.htm,56
1,T25,"Obviously, the most prominent example is the Non-core and Legacy portfolio, where we expect our wind-down efforts to result in a capital release of over 6 b...",We plan to fund this growth organically from the capital released from the unwinding of the,UBS_2023_annual_report.htm,62
2,T27,"By the end of 2024, we expect combined risk – credit and market risk risk-weighted assets to be substantially below 40 billion.","The core business-led reductions in RWA, coupled with the run-down of positions in the Non-core and Legacy business division, are expected to more than offs...",UBS_2023-Q4_report.htm,47
4,T33,"In this respect, following last year’s successful raises, we expect to issue up to 2 billion in AT1 in 2024.",,,
5,T68,"Looking ahead, we’re targeting to bring down Group HoldCo to around 90 billion by the end of 2025 while still retaining resilient buffers over regulatory mi...",,,
35,F11,"To improve efficiency of our capital stack, we intend to fund this increase by cost-effectively building-out the permissible AT1 bucket over time, bringing ...",,,
36,F12,"Specifically, we expect to reduce LRD by over 100 billion at constant FX via the wind-down of NCL and from resource optimization across our core business di...","Reduce RWA and LRD, freeing up capital for the UBS Group",UBS_2023_annual_report.htm,56
38,F22,"Going forward, we expect to operate with an LCR below our 4Q24 level of 188%, reflecting continued efforts to manage towards a more efficient funding struct...",,,


## The integration-cost total (T30)

In [6]:
forms_read = pd.read_csv(IN / "equivalent_forms_read.csv", keep_default_na=False)
t30 = forms_read[forms_read["class"] == "same target, another unit"]
t30[["unit", "file", "page", "quote", "conversion"]]

,unit,file,page,quote,conversion
829,T30,UBS_2023-Q2_report.htm,8,Cumulative integration-related expenses are expected to be broadly offset by accretion-to-par effects of approximately USD 12bn,"12 / 13 = 0.92, inside the tolerance of 10%; indirect form: the total is equated with the accretion effects"


The calls give the total of integration-related expenses as around 13bn, later around 14bn. The reports give quarterly amounts and, in the 2023-Q2 report, an indirect order of magnitude of approximately USD 12bn, published before the call figure.

## The higher capital requirement (T32)

In [7]:
corr = pd.read_csv(IN / "outcome_corrections.csv", keep_default_na=False)
c32 = corr[corr.field.str.startswith("figure_report")]
rwa_2023 = 546.5      # Group RWA at 31.12.2023 in USD bn, 2024 annual report p. 32
print("180 basis points on RWA of", rwa_2023, "bn:", round(0.018 * rwa_2023, 1), "bn")
c32[["unit", "old", "new", "evidence"]]

180 basis points on RWA of 546.5 bn: 9.8 bn


,unit,old,new,evidence
29,T32,"higher TBTF requirements phased in from end-2025 to 2030, no figure","mentioned with another figure: around USD 10bn added to the tier 1 capital requirement when fully phased in (2024-Q1 report p. 42, repeated in the 2024-Q2, ...","'We currently estimate that this will add around USD 10bn to the Group’s tier one capital requirement, when fully phased in' (UBS_2024-Q1_report.htm, page 4..."


The call gives the increase of the going concern requirement as around 180 basis points; the reports from 2024-Q1 give around USD 10bn of tier 1 capital, the same increase in another unit. T32 is therefore counted among the targets with another figure, not among the call-only figures.

## Pillar 3 reports

In [8]:
p3 = pd.read_csv(IN / "pillar3_read.csv", keep_default_na=False)
p3[["unit", "call_figure", "class_pillar3", "hits", "hits_with_figure", "quote_location"]]

,unit,call_figure,class_pillar3,hits,hits_with_figure,quote_location
0,T16,NCL operational risk RWA around 14bn by end-2026,B,36,3,"UBS_2023-Q4_pillar3.htm, page 7"
1,T25,NCL capital release over 6bn by end-2026,B,41,2,"UBS_2023-Q4_pillar3.htm, page 70"
2,T27,NCL credit and market risk RWA substantially below 40bn by end-2024,B,222,0,"UBS_2023-Q4_pillar3.htm, page 7"
3,T32,going-concern requirement +180bp to 16.7% between 2026 and 2030,C,77,0,
4,T33,AT1 issuance up to 2bn in 2024,C,166,21,
5,T68,Group HoldCo around 90bn by end-2025,C,50,1,


The six Pillar 3 reports from 2023-Q3 to 2024-Q4 state none of the figures. Three units are mentioned there in forward-looking form without a figure.

## Search in equivalent forms

In [9]:
forms = pd.read_csv(IN / "equivalent_forms.csv")
fsum = pd.read_csv(IN / "equivalent_forms_summary.csv")
per_unit = forms.groupby("unit").agg(forms=("form", "size"), examples=("form", lambda x: "; ".join(list(x)[:3])))
per_unit.join(fsum.set_index("unit")[["hits_pass_A", "hits_pass_B", "hits_pass_B_read", "distinct_passages_read"]])

,forms,examples,hits_pass_A,hits_pass_B,hits_pass_B_read,distinct_passages_read
unit,,,,,,
F11,7,ratio; increase in basis points; AT1 ratio,176,1637,146,124
F12,5,amount; share; resulting level,127,1197,203,136
F22,1,ratio,0,1156,301,82
T16,6,amount; reduction; reduction in percent,1,272,22,15
T25,4,amount; RWA equivalent at 14% CET1; effect on the CET1 ratio,0,246,4,3
T27,6,amount; total NCL RWA; share of Group RWA,156,832,126,146
T30,4,amount; amount as raised in 2024-Q4; remaining at end-2023,13,497,50,45
T33,8,amount; amount in CHF; share of RWA,18,974,70,43
T68_r1,5,amount; reduction; reduction in percent,343,2611,295,152


In [10]:
judged = pd.read_csv(IN / "equivalent_forms_judgments.csv")
eq = judged["class"].value_counts().rename("passages").to_frame()
eq.to_csv(EXPORTS / "equivalent_forms_summary.csv")
eq

,passages
class,
result,391
definition or requirement,245
other,63
other forward-looking target,58
"same target, no figure",10
"same target, another unit",1


A second search covered every equivalent form of the call figures (other units, reductions against levels, ratios against amounts), by figure and by keyword. All 768 distinct passages were read in full: one states a target in another unit (T30, above), ten state the target of their unit without a figure, and none adds a further counterpart.

In [11]:
forms_read[forms_read["class"] == "same target, no figure"][["unit", "file", "page", "quote"]]

,unit,file,page,quote
541,T25,UBS_2023_annual_report.htm,56,freeing up capital for the UBS Group
542,T25,UBS_2023_annual_report.htm,62,We plan to fund this growth organically from the capital released from the unwinding of the
543,T25,UBS_2024_annual_report.htm,52,freeing up capital for the UBS Group
561,T27,UBS_2023-Q3_report.htm,51,we expect exposures in Non-core and Legacy to reduce as a result of maturities and active unwinding of positions
587,T27,UBS_2023_annual_report.htm,56,"Reduce RWA and LRD, freeing up capital for the UBS Group"
672,T27,UBS_2024_annual_report.htm,52,We will continue to actively wind down Non-core and Legacy
696,F12,UBS_2023-Q2_report.htm,8,We intend to actively reduce the assets of the NCL business division
738,F12,UBS_2023_annual_report.htm,56,"Reduce RWA and LRD, freeing up capital for the UBS Group"
821,F12,UBS_2024_annual_report.htm,52,We will continue to actively wind down Non-core and Legacy
822,F12,UBS_2024_annual_report.htm,52,"Reduce RWA and LRD, freeing up capital for the UBS Group"


## Verification of quotes

In [12]:
quotes = pd.read_csv(IN / "quotes_all.csv")
to_2024 = quotes[~quotes.file.str.contains("2025|2026")]
check = rc.verify_quotes(to_2024, REPORTS)
check.to_csv(EXPORTS / "quote_check_to_2024.csv", index=False)
print(rc.mode_message(status))
print("quotes:", len(check), "| checked:", int(check.checked.sum()), "| verified:", int(check.verified.sum()))
check.groupby("origin").agg(quotes=("quote", "size"), checked=("checked", "sum"), verified=("verified", "sum"))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.
quotes: 129 | checked: 129 | verified: 129


,quotes,checked,verified
origin,,,
call_first.csv,5,5,5
call_vs_report_cases.csv,29,29,29
counterparts_to_2024.csv,29,29,29
equivalent_forms_read.csv,11,11,11
outcomes_units.csv,7,7,7
pillar3_read.csv,3,3,3
quotes_2025_2026.csv,3,3,3
targets_39.csv,35,35,35
targets_added_by_fls.csv,7,7,7


## Export

In [13]:
call_only.to_csv(EXPORTS / "call_only_8.csv", index=False)
targets[["unit", "call", "figure_call", "class_to_2024", "report_status", "figure_report", "file", "page"]].to_csv(EXPORTS / "report_classes_to_2024.csv", index=False)
sorted(p.name for p in EXPORTS.glob("*.csv"))

['call_only_8.csv',
 'equivalent_forms_summary.csv',
 'lead_time.csv',
 'outcomes_2025_2026.csv',
 'quote_check_to_2024.csv',
 'report_classes_to_2024.csv',
 'targets_39.csv']

In [14]:
# checks against the primary values
assert len(targets) == 39
assert targets.class_to_2024.value_counts().to_dict() == {"A": 23, "B": 12, "C": 4}
assert targets.report_status.value_counts().to_dict() == {"same figure": 23, "another figure": 7, "no figure": 4, "not mentioned": 4, "separate": 1}
assert set(call_only.unit) == {"T33", "T68", "F11", "F22", "T16", "T25", "T27", "F12"}
assert targets.set_index("unit").report_status["T30"] == "separate"
assert targets.set_index("unit").report_status["T32"] == "another figure"
assert abs(0.018 * rwa_2023 - 10) / 10 < 0.10
assert len(judged) == 768 and eq.passages["same target, another unit"] == 1 and eq.passages["same target, no figure"] == 10
assert list(t30.unit) == ["T30"]
assert (p3.hits_with_figure.astype(int) >= 0).all() and not p3.class_pillar3.eq("A").any()
assert check[check.checked].verified.all(), "a quote was not found on its page"
print("all checks passed")

all checks passed
